# 06 - Chaîne de Markov (détection de séquences)

**Idée :** pour chaque utilisateur, on regarde l'*ordre* des machines auxquelles il se connecte
(A → B → C ...). On apprend la probabilité de chaque transition pendant une période *normale*.
Ensuite, une séquence qui contient des transitions rares (ou jamais vues) est **suspecte**.

$$P(j \mid i) = \frac{N_{ij} + \alpha}{N_i + \alpha M} \qquad
\text{score} = \frac{1}{T-1}\sum_t -\log P(m_{t+1} \mid m_t)$$

- $N_{ij}$ : nombre de fois où l'utilisateur est passé de $i$ à $j$
- $\alpha$ : lissage (évite une probabilité nulle), $M$ : nombre de machines
- score **grand** = séquence **improbable** = suspect

**Entrée :** `datasets/auth_clean.csv` (créé dans le notebook 03).
Si ce fichier n'existe pas encore, le notebook utilise de **fausses données** (mode démo).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

sys.path.append(".")
import utils

os.makedirs("../results/figures", exist_ok=True)
os.makedirs("../results/metrics", exist_ok=True)

## 1. Chargement des données

In [ ]:
AUTH_PATH = "../datasets/auth_clean.csv"

if os.path.exists(AUTH_PATH):
    df = pd.read_csv(AUTH_PATH)
    print("Vraies données chargées :", len(df), "lignes")
else:
    df = utils.make_fake_logs()
    print("MODE DEMO (fausses données) :", len(df), "lignes")

df["hour_id"] = df["time"] // 3600     # numéro de l'heure depuis le début
print("Attaques (label=1) :", df["label"].sum())
df.head()

## 2. Séparation entraînement / test
On apprend le comportement **normal** avant la première attaque, puis on teste après.

In [ ]:
if df["label"].sum() > 0:
    TRAIN_END = int(df.loc[df["label"] == 1, "hour_id"].min())   # 1re heure d'attaque
else:
    TRAIN_END = int(df["hour_id"].quantile(0.7))
print("Entraînement : heures <", TRAIN_END, "| Test : heures >=", TRAIN_END)

train = df[df["hour_id"] < TRAIN_END]

## 3. Apprentissage : compter les transitions
Pour chaque utilisateur et chaque heure, on prend la séquence de machines dans l'ordre du temps
et on compte les paires (machine précédente → machine suivante).

In [ ]:
counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

for (user, h), g in train.groupby(["user", "hour_id"]):
    seq = g.sort_values("time")["dst_comp"].tolist()
    for a, b in zip(seq[:-1], seq[1:]):
        counts[user][a][b] += 1

print("Utilisateurs appris :", len(counts))

## 4. Probabilités et score
- `proba(user, a, b)` : probabilité que `user` aille de `a` vers `b`.
- `markov_score(user, seq)` : moyenne de `-log P` sur la séquence.

In [ ]:
ALPHA = 1
M = df["dst_comp"].nunique()

def proba(user, a, b):
    row = counts[user][a] if (user in counts and a in counts[user]) else {}
    n_ab = row.get(b, 0)
    n_a = sum(row.values())
    return (n_ab + ALPHA) / (n_a + ALPHA * M)

def markov_score(user, seq):
    if len(seq) < 2:
        return np.nan                       # pas de transition -> pas de score
    return float(np.mean([-np.log(proba(user, a, b)) for a, b in zip(seq[:-1], seq[1:])]))

# Exemple : les 5 transitions les plus probables d'un utilisateur
u = next(iter(counts))
a = max(counts[u], key=lambda k: sum(counts[u][k].values()))
print(f"Utilisateur {u}, depuis la machine {a} :")
for b, n in sorted(counts[u][a].items(), key=lambda x: -x[1])[:5]:
    print(f"   -> {b} : P = {proba(u, a, b):.2f}")

## 5. Scorer chaque (utilisateur, heure) de la période de test

In [ ]:
test = df[df["hour_id"] >= TRAIN_END]
rows = []
for (user, h), g in test.groupby(["user", "hour_id"]):
    seq = g.sort_values("time")["dst_comp"].tolist()
    rows.append((user, h, markov_score(user, seq), int(g["label"].max()), len(seq)))

res = pd.DataFrame(rows, columns=["user", "hour_id", "score_markov", "label", "n_events"])
res = res.dropna(subset=["score_markov"])       # on garde les heures avec >= 2 événements
print(len(res), "(utilisateur, heure) évalués")
res.nlargest(10, "score_markov")

## 6. Évaluation
Les attaques sont très rares, donc on regarde surtout la **PR-AUC** (comparée au hasard = proportion d'attaques).

In [ ]:
y, s = res["label"], res["score_markov"]
metrics = {"modele": "markov", "n_test": len(res), "n_attaques": int(y.sum()),
           "pr_auc_hasard": round(y.mean(), 4)}

if y.sum() > 0:
    metrics["pr_auc"] = round(average_precision_score(y, s), 4)
    metrics["roc_auc"] = round(roc_auc_score(y, s), 4)
    top20 = res.nlargest(20, "score_markov")
    metrics["precision@20"] = round(top20["label"].mean(), 4)

pd.DataFrame([metrics]).to_csv("../results/metrics/markov_metrics.csv", index=False)
res.to_csv("../results/metrics/markov_scores.csv", index=False)
metrics

## 7. Graphiques

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) distribution des scores
ax[0].hist(res.loc[res.label == 0, "score_markov"], bins=40, alpha=.7, label="normal", color="tab:blue")
if y.sum() > 0:
    ax[0].hist(res.loc[res.label == 1, "score_markov"], bins=40, alpha=.8, label="attaque", color="tab:red")
ax[0].set_yscale("log"); ax[0].set_xlabel("score Markov"); ax[0].set_ylabel("nombre"); ax[0].legend()
ax[0].set_title("Distribution des scores")

# (b) courbe précision-rappel
if y.sum() > 0:
    p, r, _ = precision_recall_curve(y, s)
    ax[1].plot(r, p, color="tab:purple")
    ax[1].axhline(y.mean(), ls="--", color="gray", label="hasard")
    ax[1].set_xlabel("rappel"); ax[1].set_ylabel("précision"); ax[1].legend()
    ax[1].set_title(f"Précision-rappel (PR-AUC = {metrics['pr_auc']})")

plt.tight_layout()
plt.savefig("../results/figures/markov_scores_pr.png", dpi=150)
plt.show()

## 8. Top 20 des utilisateurs les plus suspects

In [ ]:
top_users = (res.groupby("user")["score_markov"].max().sort_values(ascending=False).head(20))
plt.figure(figsize=(8, 5))
top_users[::-1].plot.barh(color="tab:purple")
plt.xlabel("score Markov maximal"); plt.title("Top 20 utilisateurs suspects (Markov)")
plt.tight_layout()
plt.savefig("../results/figures/markov_top20_users.png", dpi=150)
plt.show()

## 9. À retenir / limites
- Fonctionne bien quand chaque utilisateur a des **habitudes stables**.
- Un utilisateur sans historique d'entraînement a toujours un score élevé (faux positifs).
- On pourrait combiner ce score avec celui de l'Isolation Forest et de K-Means (notebook 08).